# 보안 Unity 카탈로그 개체

## 데모 시나리오: RetailNow – 옴니채널 소매 플랫폼
**업계 상황: 소매**
RetailNow는 미국 4개 지역(북부, 남부, 동부, 서부)에서 80개 이상의 매장과 전자상거래 플랫폼을 운영하는 옴니채널 소매업체입니다. Azure Databricks 데이터 플랫폼은 개인 정보 보호 규정에 따라 중요한 고객 및 거래 데이터를 저장합니다.

**보안 과제**: 이해관계자마다 서로 다른 수준의 액세스가 필요합니다.
- **재무팀**: 규정 준수 및 감사 보고를 위한 PII를 포함한 전체 액세스 권한
- **지역 매장 관리자**: 해당 지역의 거래만 볼 수 있습니다.
- **소매 분석가**: 전사적 쿼리 액세스가 가능하지만 고객 PII(전화, 이메일)는 볼 수 없습니다.
- **자동화된 파이프라인**: 하드코딩된 자격 증명 없이 인증

Azure Databricks 데이터 엔지니어로서 우리는 Unity Catalog를 통해 테이블을 복제하거나 수십 개의 뷰를 관리하지 않고도 소매 데이터에 대한 세밀하고 통제된 액세스를 지원하는 방법을 시연할 것입니다.

**`trainer_demo.demo_04`의 샘플 데이터:**
| Table        | Description                                                        |
|--------------|--------------------------------------------------------------------|
| `customers`  | PII(이메일, 전화번호) 및 고객 충성도 세그먼트를 포함한 고객 프로필      |
| `products`   | 가격 및 원가 마진을 포함한 제품 카탈로그                      |
| `stores`     | 지역별 오프라인 매장 위치                               |
| `transactions` | 모든 매장의 매출(고객과 제품을 연결)          |

### 설정: 샘플 소매 데이터 로드
다음 셀을 실행하여 `trainer_demo.demo_04`에 샘플 테이블을 만듭니다.
- **매장**: 북부/남부/동부/서부 지역으로 그룹화된 16개의 물리적 위치
- **제품**: 전자, 의류, 스포츠, 홈, 식품, 뷰티 전반에 걸친 20개 품목의 제품 카탈로그
- **고객**: 이메일, 전화(PII 필드), 지역 및 로열티 세그먼트를 포함한 500개의 고객 기록
- **거래**: 고객, 상품, 매장을 연결하는 판매기록 5,000건

In [0]:
from scripts.setup import setup, setup_04

spark.sql("USE CATALOG trainer_demo")
setup_04(spark)

In [0]:
%sql
-- Set working context to the retail demo schema
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Preview customer data – note the PII columns (email, phone)
SELECT * FROM customers LIMIT 5;

## 쿼리 수명 주기 이해

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.understand-query-lifecycle.png)

### 🎯 데모: 쿼리 수명 주기 관찰
주체가 Unity Catalog 테이블을 쿼리하면 요청은 데이터가 반환되기 전에 여러 보안 체크포인트를 통과합니다.
1. 클러스터 또는 SQL 웨어하우스에 **쿼리 제출**
2. **Unity 카탈로그가 주체의 권한을 검증**합니다 (카탈로그 사용 → 스키마 사용 → 선택).
3. **클라우드 자격 증명이 가정됩니다** — Unity Catalog는 단기 범위 토큰을 생성합니다.
4. 토큰을 사용하여 ADLS Gen2에서 **데이터를 읽습니다**
5. 컴퓨팅 리소스에 **행/열 필터가 적용됩니다**
6. **결과가 호출자에게 반환됩니다** — 감사 이벤트가 이미 기록되었습니다.

이 노트북을 실행하는 사람이 누구인지 확인하고 현재 데이터 컨텍스트를 확인해 보겠습니다.

In [0]:
%sql
-- Identify the current principal and context
SELECT
  current_user()    AS current_user,
  current_catalog() AS catalog,
  current_schema()  AS schema;


In [0]:
%sql
    
-- Unity Catalog records every data access in the system audit log.
-- The query below shows recent activity against our retail tables.
-- Note: requires READ privilege on system.access.audit (typically metastore admin).
SELECT
  event_time,
  user_identity.email AS user_name,
  action_name,
  request_params.commandText
FROM system.access.audit
WHERE action_name IN ('commandSubmit', 'runCommand')
  AND event_date >= CURRENT_DATE - INTERVAL 10 DAY
ORDER BY event_time DESC
LIMIT 10;

## 액세스 제어 전략 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.implement-access-control-strategies.png)

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/secure-unity-catalog-objects/media/understand-access-control-lists.png)

### 🎯 데모: 권한 부여 및 취소
RetailNow에는 데이터 액세스 요구 사항이 서로 다른 세 가지 사용자 그룹이 있습니다.
| 그룹 | 설명 | 필요 권한 |
|---|---|---|
| `retail_analysts` | 분석팀 - 비즈니스 전반에 걸친 인사이트 제공 | USE CATALOG, USE SCHEMA, SELECT on all tables (PII masked) |
| `store_managers` | 지역 관리자 - 운영 보고 | USE CATALOG, USE SCHEMA, SELECT (row-filtered per region) |
| `finance_team` | 재무 및 규정 준수 - 전체 감사 보고서 열람 가능 | USE CATALOG, USE SCHEMA, SELECT all tables including PII |

**권한 상속 알림:**

- **테이블 수준**에서 부여 = 명시적, 가장 제한적
- **SCHEMA에 대한 SELECT** 부여 = 현재 및 미래의 모든 테이블이 상속됩니다.
- **카탈로그에 대한 모든 권한** 부여 = 가장 광범위한 액세스
> **강사 참고 사항**: 그룹은 Databricks 계정 콘솔 또는 SCIM/Entra ID 동기화를 통해 생성됩니다. 아래 GRANT 문에서는 그룹이 이미 존재한다고 가정합니다.

In [0]:
%sql

CREATE GROUP retail_analysts;
CREATE GROUP finance_team;
CREATE GROUP store_managers;

In [0]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Grant traverse permissions at catalog and schema level
GRANT USE CATALOG ON CATALOG trainer_demo TO `retail_analysts`;
GRANT USE SCHEMA  ON SCHEMA  trainer_demo.demo_04 TO `retail_analysts`;

-- Grant table-level SELECT (explicit model – most restrictive)
GRANT SELECT ON TABLE trainer_demo.demo_04.customers     TO `retail_analysts`;
GRANT SELECT ON TABLE trainer_demo.demo_04.transactions  TO `retail_analysts`;
GRANT SELECT ON TABLE trainer_demo.demo_04.products      TO `retail_analysts`;
GRANT SELECT ON TABLE trainer_demo.demo_04.stores        TO `retail_analysts`;


In [0]:
%sql
-- Verify grants on the customers table
SHOW GRANTS ON TABLE trainer_demo.demo_04.customers;


In [0]:
%sql
-- Alternative: grant SELECT at schema level – inherited by ALL current and future tables.
-- Lower admin overhead, but broader access.
GRANT USE CATALOG  ON CATALOG trainer_demo        TO `finance_team`;
GRANT USE SCHEMA   ON SCHEMA  trainer_demo.demo_04 TO `finance_team`;
GRANT SELECT       ON SCHEMA  trainer_demo.demo_04 TO `finance_team`;

-- Revoke example: remove table-level access from retail_analysts for a sensitive table
-- REVOKE SELECT ON TABLE trainer_demo.demo_04.customers FROM `retail_analysts`;

-- Check all grants on the schema
SHOW GRANTS ON SCHEMA trainer_demo.demo_04;


## 세분화된 액세스 제어 이해

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.understand-fine-grained-access-control.png)

### 🎯 데모: 세분화된 액세스 제어 접근 방식 중에서 선택

Unity Catalog는 주체가 볼 수 있는 행과 열을 제한하는 두 가지 접근 방식을 제공합니다. 전체 `customers` 테이블에 SELECT를 부여하면 모든 분석가에게 PII가 노출될 수 있습니다.

**접근 방식 A – 행 및 열 보안** *(선호)*  
- 민감한 열에 마스킹 기능을 직접 첨부  
- 필터 기능을 테이블에 직접 부착  
- 모든 쿼리는 자동으로 규칙을 적용합니다. 추가 개체가 필요하지 않습니다.  
- 관리, 감사, 유지 관리할 항목이 적습니다.

**접근 방식 B – 동적 뷰**  
- `CASE WHEN is_account_group_member(...)` 로직을 사용하여 뷰 생성  
- 사용자가 뷰를 쿼리합니다. 기본 테이블은 직접 노출되지 않습니다.  
- 뷰가 여러 테이블을 조인해야 하거나 엄격한 추상화가 필요할 때 유용합니다.

RetailNow의 경우 기본 테이블의 **열 마스킹 + 행 필터링**과 대안으로 **동적 보기**를 모두 시연합니다.

## 행 필터링 및 열 마스킹 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.implement-row-filtering-column-masking.png)

### 🎯 데모: 열 마스킹 – 고객 PII 보호
`retail_analysts`은(는) 비즈니스 통찰력을 얻기 위해 `customers`에 쿼리해야 하지만 전체 전화번호나 이메일 주소를 볼 수는 없습니다. 우리는 `finance_team`의 구성원에게만 민감한 값을 공개하는 마스킹 함수를 만듭니다.

In [0]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Masking function for phone: shows last 4 digits to non-finance users
CREATE OR REPLACE FUNCTION mask_phone(phone STRING)
  RETURNS STRING
  COMMENT 'Masks phone number – finance_team sees full value; others see XXX-XXX-NNNN'
  RETURN CASE
    WHEN is_account_group_member('finance_team') THEN phone
    ELSE 'XXX-XXX-' || RIGHT(phone, 4)
  END;

-- Masking function for email: shows first 2 chars + domain to non-finance users
CREATE OR REPLACE FUNCTION mask_email(email STRING)
  RETURNS STRING
  COMMENT 'Masks email address – finance_team sees full value; others see partial'
  RETURN CASE
    WHEN is_account_group_member('finance_team') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@', SPLIT(email, '@')[1])
  END;


In [0]:
%sql
-- Attach the masking functions to the sensitive columns in customers
ALTER TABLE trainer_demo.demo_04.customers
  ALTER COLUMN phone
  SET MASK trainer_demo.demo_04.mask_phone;

ALTER TABLE trainer_demo.demo_04.customers
  ALTER COLUMN email
  SET MASK trainer_demo.demo_04.mask_email;


In [0]:
%sql
-- Query customers – phone and email are now automatically masked for non-finance users.
-- Running as admin you see real values; retail_analysts would see redacted output.
SELECT customer_id, first_name, last_name, email, phone, customer_segment, region
FROM trainer_demo.demo_04.customers
LIMIT 10;


### 🎯 데모: 행 필터링 – 지역 트랜잭션 액세스
매장 관리자는 해당 지역의 거래만 확인해야 합니다. 쿼리 런타임 시 그룹 멤버십을 평가하고 이에 따라 행을 제한하는 행 필터 함수를 만듭니다.

In [0]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Row filter function: finance_team and retail_analysts see all regions;
-- region-specific manager groups see only their region.
CREATE OR REPLACE FUNCTION filter_transaction_region(store_region STRING)
  RETURNS BOOLEAN
  COMMENT 'Restricts transaction rows to the calling principal\'s store region'
  RETURN
    is_account_group_member('finance_team')    OR
    is_account_group_member('retail_analysts') OR
    (is_account_group_member('north_store_managers') AND store_region = 'North') OR
    (is_account_group_member('south_store_managers') AND store_region = 'South') OR
    (is_account_group_member('east_store_managers')  AND store_region = 'East')  OR
    (is_account_group_member('west_store_managers')  AND store_region = 'West');

-- Attach the row filter to the transactions table
ALTER TABLE trainer_demo.demo_04.transactions
  SET ROW FILTER trainer_demo.demo_04.filter_transaction_region ON (store_region);


In [0]:
%sql
-- Query transactions with the row filter active.
-- As metastore admin you see all regions; store managers would see only their region.
SELECT
  t.transaction_id,
  c.first_name || ' ' || c.last_name AS customer_name,
  p.product_name,
  s.store_name,
  t.store_region,
  t.transaction_date,
  t.amount
FROM trainer_demo.demo_04.transactions t
JOIN trainer_demo.demo_04.customers c ON t.customer_id = c.customer_id
JOIN trainer_demo.demo_04.products  p ON t.product_id  = p.product_id
JOIN trainer_demo.demo_04.stores    s ON t.store_id    = s.store_id
ORDER BY t.transaction_date DESC
LIMIT 10;


In [0]:
%sql

ALTER TABLE trainer_demo.demo_04.transactions
  DROP ROW FILTER;

### 🎯 데모: 대안으로서의 동적 뷰
함수를 테이블에 직접 연결하는 대신 동일한 마스킹 및 필터링 논리를 캡슐화하는 **동적 보기**를 만들 수 있습니다. 사용자에게 뷰에 대한 액세스 권한이 부여됩니다. 기본 테이블은 숨겨진 상태로 유지됩니다.

이 접근 방식은 다음과 같은 경우에 유용합니다:
- 일관된 보안 규칙으로 여러 테이블을 조인해야 합니다.
- 조직 정책에 따라 소비자와 소스 테이블을 엄격하게 분리해야 합니다.

In [0]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_04;

-- Dynamic view: masks PII and restricts rows based on group membership
CREATE OR REPLACE VIEW vw_customers_secure AS
SELECT
  customer_id,
  first_name,
  last_name,
  CASE
    WHEN is_account_group_member('finance_team') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@', SPLIT(email, '@')[1])
  END AS email,
  CASE
    WHEN is_account_group_member('finance_team') THEN phone
    ELSE 'XXX-XXX-' || RIGHT(phone, 4)
  END AS phone,
  region,
  customer_segment
FROM trainer_demo.demo_04.customers
WHERE
  -- finance_team and retail_analysts see all customers;
  -- store managers see only customers in their region
  CASE
    WHEN is_account_group_member('finance_team')    THEN TRUE
    WHEN is_account_group_member('retail_analysts') THEN TRUE
    WHEN is_account_group_member('north_store_managers') AND region = 'North' THEN TRUE
    WHEN is_account_group_member('south_store_managers') AND region = 'South' THEN TRUE
    WHEN is_account_group_member('east_store_managers')  AND region = 'East'  THEN TRUE
    WHEN is_account_group_member('west_store_managers')  AND region = 'West'  THEN TRUE
    ELSE FALSE
  END;

-- Query the secure view
SELECT * FROM vw_customers_secure LIMIT 10;


## Azure Key Vault 비밀에 액세스

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.access-azure-key-vault.png)

### 🎯 데모: Azure Key Vault 비밀에 액세스

RetailNow의 데이터 플랫폼은 여러 외부 시스템(충성도 PostgreSQL 데이터베이스, ERP 시스템(Azure SQL 데이터베이스) 등)에 연결됩니다. 자격 증명은 노트북 코드 또는 작업 구성에 표시되어서는 안 됩니다.

**패턴: Azure Key Vault 지원 비밀 범위**
1. 비밀은 **Azure Key Vault**(보안팀에서 관리)에 있습니다.
2. Databricks의 **비밀 범위**는 Key Vault를 참조합니다.
3. 코드는 `dbutils.secrets.get()`을 사용하여 런타임에 값을 검색합니다. 출력은 항상 **[편집됨]**입니다.

**일반적인 소매 사용 사례:**
- 충성도/ERP 데이터베이스를 위한 JDBC 연결 문자열
- 결제 처리자 또는 제3자 데이터 피드용 API 키
- 파트너 파일 삭제를 위한 스토리지 계정 키

In [0]:
# List all secret scopes registered in this workspace
# In a real setup you might have scopes like 'retail-db-secrets', 'payment-api-keys'
# scopes = dbutils.secrets.listScopes()
# for s in scopes:
#    print(f"Scope: {s.name}")

# ------------------------------------------------------------
# Retrieve a secret – the value is ALWAYS redacted in output
# ------------------------------------------------------------
# db_password = dbutils.secrets.get(scope="retail-db-secrets", key="loyalty-db-password")
# print(db_password)   # prints: [REDACTED]

# Use in a JDBC connection to the loyalty database
# loyalty_df = (
#     spark.read.format("jdbc")
#     .option("url", "jdbc:postgresql://loyalty-db.retailnow.com:5432/loyalty")
#     .option("dbtable", "loyalty_points")
#     .option("user",     dbutils.secrets.get("retail-db-secrets", "loyalty-db-user"))
#     .option("password", dbutils.secrets.get("retail-db-secrets", "loyalty-db-password"))
#     .load()
# )

# List keys in a scope (returns metadata – NOT secret values)
# keys = dbutils.secrets.list("retail-db-secrets")
# for k in keys: print(k.key)

db_password = dbutils.secrets.get(scope="retail-kv-scope", key="loyalty-api-key")
print(db_password)  

# print("Secrets are always [REDACTED] in notebook output – safe to share notebooks.")


In [0]:
%sql
-- In Databricks SQL use the secret() function – same redaction behaviour.
-- Example: reference credentials inside a CREATE TABLE / OPTIONS block.

-- CREATE TABLE IF NOT EXISTS trainer_demo.demo_04.erp_orders
-- USING jdbc
-- OPTIONS (
--   url      'jdbc:sqlserver://erp.retailnow.com:1433',
--   dbtable  'orders',
--   user     secret('retail-db-secrets', 'erp-db-user'),
--   password secret('retail-db-secrets', 'erp-db-password')
-- );

SELECT 'Use secret(scope, key) in SQL – output is [REDACTED]' AS note;


## 서비스 주체를 사용하여 데이터 액세스 인증

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.authenticate-data-access-service-principals.png)

### 🎯 데모: 서비스 주체로 데이터 액세스 인증
RetailNow의 원시 POS 내보내기 파일은 ADLS Gen2 계정에 저장되어 있으며, 아직 Unity 카탈로그에 등록되지 않았습니다. 야간 ETL 작업은 이러한 파일을 읽기 위해 OAuth2로 구성된 **서비스 주체**를 사용해야 합니다.

**주요 보안 관행**: 클라이언트 암호는 **항상** Databricks 비밀 범위에서 검색되며 하드코딩되지 않습니다.

> **강사 참고 사항**: 이 셀은 개념적입니다. 실제 서비스 주체, 스토리지 계정 및 비밀 범위가 필요합니다. 코드 패턴은 프로덕션 사용을 위한 모범 사례를 보여줍니다.

In [0]:
# ── Service Principal Authentication for ADLS Gen2 ───────────────────────────
# Step 1: Retrieve the client secret from the secret scope (never hard-code it)
# sp_client_secret = dbutils.secrets.get(scope="retail-sp-secrets", key="sp-client-secret")

storage_account  = "retailnowdatalake"
sp_application_id = "<application-client-id>"   # replace with actual App (client) ID
sp_tenant_id      = "<directory-tenant-id>"      # replace with actual Directory (tenant) ID

# Step 2: Configure Spark for OAuth2 client credentials flow
# spark.conf.set(
#     f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
# spark.conf.set(
#     f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
#     "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
#     sp_application_id)
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
#     sp_client_secret)
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
#     f"https://login.microsoftonline.com/{sp_tenant_id}/oauth2/token")

# Step 3: Read raw POS exports using the ABFS driver
# pos_df = spark.read.csv(
#     f"abfss://raw@{storage_account}.dfs.core.windows.net/pos-exports/",
#     header=True, inferSchema=True
# )
# pos_df.show(5)

print("Service principal config pattern: credentials come from secret scope, never hardcoded.")
print("Note: Microsoft recommends managed identities for Unity Catalog storage credentials.")


## 관리 ID로 리소스 액세스 인증

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/secure-unity-catalog-objects.authenticate-resource-access-managed-identities.png)

### 🎯 데모: 관리 ID로 리소스 액세스 인증

프로덕션 환경에서 **관리 ID**는 Unity Catalog 스토리지 자격 증명에 대해 선호되는 인증 방법입니다. 서비스 주체 대비 주요 이점은 다음과 같습니다.

- **관리할 비밀이 없음**: Azure가 자격 증명 교체를 자동으로 처리합니다.
- **네트워크 제한 저장소 지원**: 관리 ID는 방화벽 뒤의 ADLS 계정에 액세스할 수 있습니다. 서비스 주체는 불가능합니다.
- **더 단순한 거버넌스**: 감사 및 순환할 자격 증명이 적습니다.

**RetailNow 시나리오**: Azure Databricks Access Connector는 시스템 할당 관리 ID를 사용하여 배포됩니다. 이 ID에는 소매 데이터 레이크에 **Storage Blob 데이터 기여자** 권한이 부여됩니다. Unity 카탈로그 저장소 자격 증명은 커넥터를 참조하고, 외부 위치는 특정 컨테이너 경로를 노출합니다.

> **트레이너 참고사항**: 아래 SQL 명령에는 메타스토어 관리자 권한이 필요합니다. 개념적으로 또는 사전 구성된 관리 환경에서 시연합니다.

In [0]:
%sql
-- ── Step 1: Create a storage credential (metastore admin required) ─────────
-- CREATE STORAGE CREDENTIAL retailnow_adls_credential
--   WITH AZURE MANAGED IDENTITY
--   (DIRECTORY '/subscriptions/<sub-id>/resourceGroups/retailnow-rg/providers/Microsoft.Databricks/accessConnectors/retailnow-connector');

-- ── Step 2: Create an external location pointing to ADLS Gen2 ─────────────
-- CREATE EXTERNAL LOCATION IF NOT EXISTS retailnow_raw_data
--   URL 'abfss://raw@retailnowdatalake.dfs.core.windows.net/'
--   WITH (CREDENTIAL retailnow_adls_credential)
--   COMMENT 'Raw POS and e-commerce data exports for RetailNow';

-- ── Step 3: Grant data engineers access ───────────────────────────────────
-- GRANT READ FILES  ON EXTERNAL LOCATION retailnow_raw_data TO `data_engineers`;
-- GRANT WRITE FILES ON EXTERNAL LOCATION retailnow_raw_data TO `data_engineers`;

-- List existing storage credentials and external locations in the metastore
SHOW STORAGE CREDENTIALS;


In [0]:
%sql
-- List all external locations registered in the metastore
SHOW EXTERNAL LOCATIONS;


---
## 데모 요약
이 데모에서는 Unity Catalog를 사용하여 **RetailNow** 옴니채널 소매 플랫폼을 보호했습니다.
 Topic | Demonstrated |
---|---|
 Query lifecycle | How Unity Catalog intercepts, validates, logs, and scopes every query |
 Access control strategies | GRANT / REVOKE at catalog, schema, and table level; inheritance model |
 Fine-grained access control | Row & Column Security vs Dynamic Views – trade-offs and when to use each |
 Row filtering | `filter_transaction_region` restricts transaction rows per store region |
 Column masking | `mask_phone` and `mask_email` functions attached to `customers` |
 Azure Key Vault secrets | `dbutils.secrets.get()` and `secret()` SQL function – always [REDACTED] |
 Service principal auth | OAuth2 Spark configuration – secret from scope, never hardcoded |
 Managed identity auth | Storage credential + external location via Access Connector |

**주요 보안 원칙:**  
1. **자격 증명을 하드코딩하지 마세요** - 런타임 시 항상 비밀 범위에서 검색합니다.  
2. **최소 권한** – 가능한 가장 낮은 수준에서 필요한 최소 권한만 부여합니다.  
3. **마스크, 중복하지 않음** – 테이블 복사본에 대한 열 마스킹 및 행 필터링  
4. **관리형 ID 선호** – 프로덕션 환경의 Unity Catalog 스토리지 자격 증명용  
5. **감사 로그 신뢰** – `system.access.audit`은 모든 액세스 이벤트를 캡처합니다.